## Part 1: Parsing a Sentence with CKY

Given the following grammar in CNF.
1. Write the code for CKY algorithm and verify if the **sentence** follows or can be parsed by this grammar or not.
2. Also print the final CKY chart/table.

In [ ]:
import nltk
from nltk.parse import ChartParser
from nltk.grammar import CFG

In [ ]:
cky_grammar = CFG.fromstring("""
    S -> NP VP
    NP -> Det N
    VP -> V NP
    Det -> 'the'
    N -> 'man' | 'dog'
    V -> 'saw'
""")

sentence = "the man saw the dog"

In [5]:
def cky_parse(sentence, grammar, start_symbol="S"):
    words = sentence.split()
    n = len(words)
    chart = [[set() for _ in range(n + 1)] for _ in range(n)]
    for i, word in enumerate(words):
        for production in grammar.productions():
            rhs = production.rhs()

            if len(rhs) == 1 and rhs[0] == word:
                chart[i][i + 1].add(production.lhs().symbol())
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length
            for k in range(i + 1, j):
                for production in grammar.productions():
                    rhs = production.rhs()
                    if len(rhs) == 2:
                        B = rhs[0].symbol()
                        C = rhs[1].symbol()
                        A = production.lhs().symbol()
                        if B in chart[i][k] and C in chart[k][j]:
                            chart[i][j].add(A)
    print("Final CKY Chart:")
    for i in range(n):
        for j in range(i + 1, n + 1):
            if chart[i][j]:
                print(f"chart[{i}][{j}] = {chart[i][j]}")
    result = start_symbol in chart[0][n]
    print("\nCan be parsed?", result)
    return chart, result

chart, result = cky_parse(sentence, cky_grammar)

Final CKY Chart:
chart[0][1] = {'Det'}
chart[0][2] = {'NP'}
chart[0][5] = {'S'}
chart[1][2] = {'N'}
chart[2][3] = {'V'}
chart[2][5] = {'VP'}
chart[3][4] = {'Det'}
chart[3][5] = {'NP'}
chart[4][5] = {'N'}

Can be parsed? True


## Part 2: Comparision with NLTK's chart parser
After implementing your CKY algorithm, compare its output for the given sentence with the output of NLTK's built-in chart parser using the `cky_grammar`. This comparison will help you verify if your implementation is correct.

In [6]:
# NLTK Chart Parser

parser = ChartParser(cky_grammar)

trees = list(parser.parse(sentence.split()))

print("Number of parse trees:", len(trees))

for tree in trees:
    print(tree)

Number of parse trees: 1
(S (NP (Det the) (N man)) (VP (V saw) (NP (Det the) (N dog))))
